# 神经网络分类算法设计与训练

**目的：** 基于 TF-IDF 特征训练神经网络分类器，使用 KMeans 聚类标签作为训练目标，保存模型到 models/

In [ ]:
import pymysql
import pandas as pd
import numpy as np
import pickle
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')

In [ ]:
# ====== 加载 TF-IDF 特征和聚类标签 ======

# 1. 加载之前训练的向量器
model_dir = '../models'
with open(f'{model_dir}/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)

# 2. 从 MySQL 读取数据并提取特征
conn = pymysql.connect(
    host='localhost', port=3306, user='root',
    password='123456', database='recruitment_ai', charset='utf8mb4'
)
df = pd.read_sql('SELECT * FROM job_listing', conn)
conn.close()

# 3. 读聚类标签（从 cluster_result 表或从 KMeans notebook 的变量中获取）
conn = pymysql.connect(
    host='localhost', port=3306, user='root',
    password='123456', database='recruitment_ai', charset='utf8mb4'
)
cluster_df = pd.read_sql('SELECT job_id, cluster_label FROM cluster_result', conn)
conn.close()

# 合并标签
df = df.merge(cluster_df, left_on='id', right_on='job_id', how='inner')
print(f'有聚类标签的数据: {len(df)} 条')
print(f'标签分布: {dict(df["cluster_label"].value_counts().sort_index())}')

In [ ]:
# ====== 文本预处理（与聚类 notebook 保持一致）=====
import jieba
import re

stopwords = {'的', '了', '和', '是', '就', '都', '而', '及', '与', '着', '或', '一个', '没有'}

def preprocess(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9\s]', ' ', str(text))
    words = jieba.cut(text)
    return ' '.join(w for w in words if len(w) > 1 and w not in stopwords)

df['text_feature'] = (df['skills_raw'].fillna('') + ' ' + df['job_title'].fillna('')).apply(preprocess)
X = tfidf.transform(df['text_feature']).toarray()
y = df['cluster_label'].values
print(f'特征矩阵: {X.shape}, 标签: {y.shape}')

In [ ]:
# ====== 数据集划分 ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 转 PyTorch Tensor
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=32)

print(f'训练集: {len(train_ds)}, 测试集: {len(test_ds)}')

In [ ]:
# ====== 神经网络模型 ======
class RecruitmentClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)

INPUT_DIM = X.shape[1]
HIDDEN_DIM = 256
NUM_CLASSES = len(set(y))
model = RecruitmentClassifier(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES)
print(f'模型参数量: {sum(p.numel() for p in model.parameters()):,}')
print(f'输入维度={INPUT_DIM} 隐藏维度={HIDDEN_DIM} 类别数={NUM_CLASSES}')

In [ ]:
# ====== 训练 ======
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 50
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in test_dl:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        acc = correct / total
        print(f'Epoch {epoch+1:3d} | Loss: {total_loss/len(train_dl):.4f} | Test Acc: {acc:.4f}')

In [ ]:
# ====== 最终评估 ======
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb).argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(yb.cpu().tolist())

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(all_labels, all_preds, zero_division=0))
print('混淆矩阵:')
print(confusion_matrix(all_labels, all_preds))

In [ ]:
# ====== 保存模型 ======
os.makedirs(model_dir, exist_ok=True)

checkpoint = {
    'state_dict': model.state_dict(),
    'input_dim': INPUT_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_classes': NUM_CLASSES,
}
torch.save(checkpoint, f'{model_dir}/nn_classifier.pt')

# 保存标准化器
with open(f'{model_dir}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f'模型已保存: {model_dir}/nn_classifier.pt')
print(f'标准化器已保存: {model_dir}/scaler.pkl')